In [23]:
import os
import numpy as np 
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from collections import Counter , defaultdict
import random
from embedding_extraction import extract_embedding


In [6]:
data_dir = '/home/waarab/ia/free-gen-playlist/data/jamendo'

In [7]:
#TOP 50 TAGS

top_50_tags = [
    "genre---alternative",
    "genre---ambient",
    "genre---atmospheric",
    "genre---chillout",
    "genre---classical",
    "genre---dance",
    "genre---downtempo",
    "genre---easylistening",
    "genre---electronic",
    "genre---experimental",
    "genre---folk",
    "genre---funk",
    "genre---hiphop",
    "genre---house",
    "genre---indie",
    "genre---instrumentalpop",
    "genre---jazz",
    "genre---lounge",
    "genre---metal",
    "genre---newage",
    "genre---orchestral",
    "genre---pop",
    "genre---popfolk",
    "genre---poprock",
    "genre---reggae",
    "genre---rock",
    "genre---soundtrack",
    "genre---techno",
    "genre---trance",
    "genre---triphop",
    "genre---world",
    "instrument---acousticguitar",
    "instrument---bass",
    "instrument---computer",
    "instrument---drummachine",
    "instrument---drums",
    "instrument---electricguitar",
    "instrument---electricpiano",
    "instrument---guitar",
    "instrument---keyboard",
    "instrument---piano",
    "instrument---strings",
    "instrument---synthesizer",
    "instrument---violin",
    "instrument---voice",
    "mood/theme---emotional",
    "mood/theme---energetic",
    "mood/theme---film",
    "mood/theme---happy",
    "mood/theme---relaxing",
]


In [8]:
def get_path(path):
    df = pd.read_csv("/home/waarab/ia/free-gen-playlist/mtg-jamendo-dataset/data/splits/"+path,
        sep="\t",
        engine="python",
        on_bad_lines="skip"  # or "warn"
    )
    
    return df

#Get Paths and tags for a specific type ( genre, emotion, instrument ). Get only the top 50 tags and only those with one tag
def get_paths_tags(type_classif):
    
    splits = [ "split-0/" , "split-1/","split-2/"]
    paths = []
    tags = []
    for split in splits :
        df = get_path(split+"autotagging_" + type_classif+"-train.tsv")
        one_tag_rows = df[df["TAGS"].apply(lambda t: isinstance(t, str) and len(t.split()) == 1 and (t in top_50_tags))]
        
        paths.extend(one_tag_rows["PATH"].tolist())
        
        tags.extend(one_tag_rows["TAGS"].apply(lambda t: t.split("---")[-1]).tolist())
    
    return paths , tags 


In [9]:
genre_paths , genre_tags = get_paths_tags("genre")

assert(len(genre_paths)== len(genre_tags))

print(f"The number of music files : {len(genre_paths)}\n")

print(genre_paths[:5])
print(genre_tags[:5])

The number of music files : 20115

['41/241.mp3', '42/242.mp3', '43/243.mp3', '44/244.mp3', '45/245.mp3']
['rock', 'rock', 'rock', 'rock', 'rock']


In [10]:
counter = Counter(genre_tags)
print(counter)


Counter({'electronic': 4283, 'classical': 3022, 'pop': 1650, 'reggae': 1459, 'rock': 1253, 'soundtrack': 1143, 'hiphop': 823, 'ambient': 810, 'jazz': 737, 'poprock': 735, 'experimental': 728, 'metal': 705, 'easylistening': 492, 'techno': 315, 'newage': 296, 'trance': 274, 'folk': 246, 'alternative': 223, 'world': 202, 'house': 140, 'indie': 122, 'funk': 103, 'popfolk': 63, 'chillout': 61, 'triphop': 58, 'dance': 57, 'orchestral': 50, 'instrumentalpop': 31, 'atmospheric': 20, 'lounge': 11, 'downtempo': 3})


In [11]:


def balance_dataset_with_reference(paths, tags, reference_class="world", seed=42):
    random.seed(seed)

    counter = Counter(tags)
    target_count = counter[reference_class]
    print(f"Target samples per class = {target_count}")

    # Grouper indices par classe
    class_to_indices = defaultdict(list)
    for i, t in enumerate(tags):
        class_to_indices[t].append(i)

    selected_indices = []

    for cls, idxs in class_to_indices.items():
        if len(idxs) >= target_count:
            selected_indices.extend(random.sample(idxs, target_count))
        # else: on ignore les classes < target_count

    selected_indices = sorted(selected_indices)

    balanced_paths = [paths[i] for i in selected_indices]
    balanced_tags = [tags[i] for i in selected_indices]

    return balanced_paths, balanced_tags


In [12]:
balanced_genre_paths, balanced_genre_tags = balance_dataset_with_reference(
    genre_paths, genre_tags
)



print(Counter(balanced_genre_tags))


Target samples per class = 202
Counter({'rock': 202, 'classical': 202, 'pop': 202, 'electronic': 202, 'soundtrack': 202, 'ambient': 202, 'world': 202, 'jazz': 202, 'trance': 202, 'newage': 202, 'metal': 202, 'alternative': 202, 'easylistening': 202, 'hiphop': 202, 'reggae': 202, 'experimental': 202, 'folk': 202, 'techno': 202, 'poprock': 202})


In [13]:
# Generate embeddings for the type "type_classif" using the audio mentionned in "paths" with "tags"
def generate_embeddings(type_classif ,paths , tags):
    
    embeddings = []
    filenames = []
    filetag = []
    
    
    paths = [path.split('.')[0]+".low.mp3" for path in paths] # we are working with .low.mp3 files 
    
    for i, (p, tag) in enumerate(zip(paths, tags), 1):    
        full_path = os.path.join(data_dir, p)
        
        # Extraire embedding
        embedding = extract_embedding(full_path)  # doit renvoyer un np.array
        
        embeddings.append(embedding)
        filenames.append(p)
        filetag.append(tag)
    
        print(f"✅ fichiers traités : {p} , {type_classif}: {tag} ")
        
        # Libérer GPU si extract_embedding utilise torch
        torch.cuda.empty_cache()
    
    # Convertir en numpy array
    embeddings = np.array(embeddings)
    
    # Créer DataFrame
    df = pd.DataFrame(embeddings)
    df['filename'] = filenames
    df[type_classif] = filetag
    csv_path = "muq_embedding_jamendo_"+type_classif+".csv"
    df.to_csv(csv_path, index=False)
    print(f"✅ Extraction terminée ! file saved to : {csv_path}")


## T-SNE Visualisation

In [14]:
def tsn_visualisation(type_classif, labeled_csvs, unlabeled_csvs=None):

    if unlabeled_csvs is None:
        unlabeled_csvs = []

    embeddings_list = []
    tags_list = []
    source_mask = []  # True = labeled, False = unlabeled

    # ======================
    # CSV AVEC LABELS
    # ======================
    for csv_path in labeled_csvs:
        data = pd.read_csv(csv_path)

        tags = data[type_classif].to_numpy()

        drop_cols = [type_classif]
        if 'filename' in data.columns:
            drop_cols.append('filename')

        embeddings = data.drop(columns=drop_cols).to_numpy()

        embeddings_list.append(embeddings)
        tags_list.append(tags)
        source_mask.extend([True] * len(embeddings))

    # ======================
    # CSV SANS LABELS (DROP DES 2 DERNIÈRES COLONNES)
    # ======================
    for csv_path in unlabeled_csvs:
        data = pd.read_csv(csv_path)


        # Drop systématiquement les 2 dernières colonnes (labels)
        data = data.iloc[:, :-1]

        embeddings = data.to_numpy()

        embeddings_list.append(embeddings)
        tags_list.append(np.array([None] * len(embeddings)))
        source_mask.extend([False] * len(embeddings))

    # ======================
    # CONCATÉNATION
    # ======================
    embeddings = np.vstack(embeddings_list)
    tags = np.concatenate(tags_list)
    source_mask = np.array(source_mask)

    print("Total embeddings:", embeddings.shape)

    # ======================
    # t-SNE
    # ======================
    X_2d = TSNE(n_components=2, random_state=42).fit_transform(embeddings)

    # ======================
    # PLOT
    # ======================
    plt.figure(figsize=(10, 8))

    # --- Unlabeled (projection only)
    unlabeled_idx = ~source_mask
    if np.any(unlabeled_idx):
        plt.scatter(
            X_2d[unlabeled_idx, 0],
            X_2d[unlabeled_idx, 1],
            color="lightgray",
            alpha=0.3,
            s=15,
            label="Projection only"
        )

    # --- Labeled
    labeled_idx = source_mask
    labeled_tags = tags[labeled_idx]
    unique_tags = np.unique(labeled_tags)

    cmap = plt.get_cmap("tab20")
    colors = {tag: cmap(i / len(unique_tags)) for i, tag in enumerate(unique_tags)}

    for tag in unique_tags:
        idx = (tags == tag) & labeled_idx
        plt.scatter(
            X_2d[idx, 0],
            X_2d[idx, 1],
            label=str(tag),
            color=colors[tag],
            alpha=0.8,
            s=20
        )

    plt.title(f"t-SNE Visualization of MuQ Embeddings ({type_classif})")
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.legend()
    plt.grid(True)

    plt.savefig(f"tsne_muQ_embeddings_{type_classif}.png", dpi=300)
    print(f"✅ t-SNE plot saved as tsne_muQ_embeddings_{type_classif}.png")

    return X_2d


## Extract emotions Dataset

In [15]:
emotion_paths , emotion_tags = get_paths_tags("moodtheme")

assert(len(emotion_paths)== len(emotion_tags))

print(f"The number of music files : {len(emotion_paths)}\n")

print(emotion_paths[:5])
print(emotion_tags[:5])

The number of music files : 3843

['65/2165.mp3', '47/6247.mp3', '91/7391.mp3', '13/14013.mp3', '14/14014.mp3']
['film', 'emotional', 'emotional', 'relaxing', 'relaxing']


In [16]:
counter_emotion = Counter(emotion_tags)
print(counter_emotion)

Counter({'energetic': 1039, 'film': 988, 'relaxing': 878, 'happy': 473, 'emotional': 465})


In [17]:
emotion_balanced_path , emotion_balanced_tags = balance_dataset_with_reference(paths= emotion_paths , tags=emotion_tags ,reference_class='emotional')

print(Counter(emotion_balanced_tags))

Target samples per class = 465
Counter({'film': 465, 'emotional': 465, 'relaxing': 465, 'happy': 465, 'energetic': 465})


## Extract Instrument Dataset

In [18]:
instrument_paths , instrument_tags = get_paths_tags("instrument")

print(f"the size of instrument dataSet :{ len(instrument_paths)}")

print(instrument_paths[:5])
print(instrument_tags[:5])

the size of instrument dataSet :16348
['82/382.mp3', '83/383.mp3', '84/384.mp3', '86/386.mp3', '87/387.mp3']
['voice', 'voice', 'voice', 'voice', 'voice']


In [19]:
print(Counter(instrument_tags))

Counter({'piano': 5736, 'synthesizer': 4049, 'guitar': 2986, 'computer': 686, 'electricguitar': 502, 'acousticguitar': 492, 'bass': 468, 'strings': 364, 'drums': 274, 'voice': 227, 'violin': 176, 'keyboard': 141, 'electricpiano': 131, 'drummachine': 116})


In [20]:
instrument_balanced_paths , instrument_balanced_tags = balance_dataset_with_reference(instrument_paths, instrument_tags,'voice')

print(Counter(instrument_balanced_tags))

Target samples per class = 227
Counter({'voice': 227, 'synthesizer': 227, 'piano': 227, 'electricguitar': 227, 'guitar': 227, 'drums': 227, 'bass': 227, 'acousticguitar': 227, 'strings': 227, 'computer': 227})


# Extract Embeddings

In [21]:
# Generate embeddings for the type "type_classif" using the audio mentionned in "paths" with "tags"
def generate_embeddings(type_classif ,paths , tags):
    
    embeddings = []
    filenames = []
    filetag = []
    
    
    paths = [path.split('.')[0]+".low.mp3" for path in paths] # we are working with .low.mp3 files 
    
    for i, (p, tag) in enumerate(zip(paths, tags), 1):    
        full_path = os.path.join(data_dir, p)
        
        # Extraire embedding
        embedding = extract_embedding(full_path)  # doit renvoyer un np.array
        
        embeddings.append(embedding)
        filenames.append(p)
        filetag.append(tag)
    
        print(f"✅ fichiers traités : {p} , {type_classif}: {tag} ")
        
        # Libérer GPU si extract_embedding utilise torch
        torch.cuda.empty_cache()
    
    # Convertir en numpy array
    embeddings = np.array(embeddings)
    
    # Créer DataFrame
    df = pd.DataFrame(embeddings)
    df['filename'] = filenames
    df[type_classif] = filetag
    csv_path = "muq_embedding_jamendo_"+type_classif+".csv"
    df.to_csv(csv_path, index=False)
    print(f"✅ Extraction terminée ! file saved to : {csv_path}")


In [22]:
generate_embeddings("emotion",emotion_balanced_path ,emotion_balanced_tags)
generate_embeddings("genre",balanced_genre_paths ,balanced_genre_tags)
generate_embeddings("instrument",instrument_balanced_paths ,instrument_balanced_tags)

✅ fichiers traités : 65/2165.low.mp3 , emotion: film 
✅ fichiers traités : 47/6247.low.mp3 , emotion: emotional 
✅ fichiers traités : 91/7391.low.mp3 , emotion: emotional 
✅ fichiers traités : 14/14014.low.mp3 , emotion: relaxing 
✅ fichiers traités : 15/14015.low.mp3 , emotion: relaxing 
✅ fichiers traités : 20/14020.low.mp3 , emotion: relaxing 
✅ fichiers traités : 21/14021.low.mp3 , emotion: relaxing 
✅ fichiers traités : 22/14022.low.mp3 , emotion: relaxing 
✅ fichiers traités : 25/14025.low.mp3 , emotion: relaxing 
✅ fichiers traités : 27/14027.low.mp3 , emotion: relaxing 
✅ fichiers traités : 28/14028.low.mp3 , emotion: relaxing 
✅ fichiers traités : 17/25217.low.mp3 , emotion: emotional 
✅ fichiers traités : 20/25220.low.mp3 , emotion: happy 
✅ fichiers traités : 37/25237.low.mp3 , emotion: happy 
✅ fichiers traités : 41/25241.low.mp3 , emotion: emotional 
✅ fichiers traités : 93/28193.low.mp3 , emotion: film 
✅ fichiers traités : 96/34696.low.mp3 , emotion: film 
✅ fichiers tra